**Goal**:
- get hotel name from the xlsx file.
- search query the hotel name via the GET api/v1/hotels/searchDestinationOrHotel api
- get the entityid
- go to GET api/v1/hotels/getHotelDetails
- enter the entityid as parameter for 
    - hotelId
    - entityId
- 
check the policies.content part of the json file.
    - is it queen-or-king-bed-included
    - does it have breakfast-included
    - does it have pool-access-included
    - does it have wifi-included

```bash
curl --request GET \
	--url 'https://sky-scrapper.p.rapidapi.com/api/v1/hotels/searchDestinationOrHotel?query=RedDoorz%20Plus' \
	--header 'x-rapidapi-host: sky-scrapper.p.rapidapi.com' \
	--header 'x-rapidapi-key: f3068fa39amsha256be6eec9ae0ep135038jsnbf15340e581c'
```

Let's start by adding our pip installs

In [ ]:
%pip install requests pandas python-dotenv
%pip install jupyter
%pip install playwright
%pip install -U typing_extensions


# Helper Functions
- To Start, I'll be adding helper functions which are used to replace values or to print JSON dictionaries.

In [ ]:
# HELPER FUNCTIONS
import json
import math
import requests

def prettyPrintJSON(data: object) -> None:
    print(json.dumps(data, indent=2, ensure_ascii=False))

def replace_space_with_percent20(value: str) -> str:
    return value.replace(" ", "%20")

REGION_KEYWORDS: dict[str, list[str]] = {
    "NCR": [
        "quezon city", "manila", "makati", "taguig", "pasig", "paranaque",
        "parañaque", "pasay", "caloocan", "mandaluyong", "marikina",
        "san juan", "valenzuela", "las pinas", "las piñas", "muntinlupa",
        "navotas", "malabon", "pateros",
    ],
    "CAR": [
        "abra", "apayao", "benguet", "ifugao", "kalinga",
        "mountain province", "baguio",
    ],
    "Region I": [
        "ilocos norte", "ilocos sur", "la union", "pangasinan",
    ],
    "Region II": [
        "cagayan", "isabela", "nueva vizcaya", "quirino", "tuguegarao",
    ],
    "Region III": [
        "bulacan", "pampanga", "tarlac", "zambales", "bataan",
        "nueva ecija", "olongapo", "angeles", "aurora",
    ],
    "Region IV-A": [
        "cavite", "laguna", "los banos", "los baños", "batangas",
        "lipa city", "rizal", "quezon province", "lucena", "tayabas",
        "atimonan", "candelaria", "ibabang tayuman", "mauban",
        "pagbilao", "infanta",
    ],
    "Region IV-B": [
        "mindoro", "marinduque", "romblon", "palawan", "coron",
        "el nido", "puerto princesa",
    ],
    "Region V": [
        "albay", "camarines", "sorsogon", "catanduanes", "masbate",
        "legazpi", "naga", "naga city",
    ],
    "Region VI": [
        "iloilo", "antique", "aklan", "boracay", "malay", "capiz",
        "roxas city", "guimaras", "bacolod", "negros occidental",
        "san carlos city", "buruanga",
    ],
    "Region VII": [
        "cebu", "bohol", "negros oriental", "mandaue", "lapu-lapu",
        "samboan", "dumaguete", "siquijor",
    ],
    "Region VIII": [
        "leyte", "samar", "tacloban", "biliran",
    ],
    "NIR": [
        "negros occidental", "negros oriental",
    ],
    "Region IX": [
        "zamboanga", "dipolog", "pagadian",
    ],
    "Region X": [
        "cagayan de oro", "misamis", "bukidnon", "camiguin",
        "iligan", "iligan city",
    ],
    "Region XI": [
        "davao", "digos", "mati", "tagum",
    ],
    "Region XII": [
        "south cotabato", "sultan kudarat", "sarangani", "general santos",
    ],
    "Region XIII": [
        "agusan", "surigao", "butuan", "dinagat",
    ],
    "BARMM": [
        "basilan", "sulu", "tawi-tawi", "maguindanao", "lanao del sur",
    ],
}

def get_ph_region(exact_location: str) -> str:
    if not exact_location:
        return "Unknown"

    normalized = exact_location.casefold()

    for region_name, keywords in REGION_KEYWORDS.items():
        for keyword in keywords:
            if keyword.casefold() in normalized:
                return region_name

    return "Unknown"

def get_exact_location_from_lat_lng(
    latitude: float,
    longitude: float,
    timeout: int = 15,
) -> str | None:
    url = "https://nominatim.openstreetmap.org/reverse"
    params: dict[str, float | str] = {
        "lat": latitude,
        "lon": longitude,
        "format": "jsonv2",
    }
    headers = {
        "User-Agent": "scrape-hotel-prices-project/1.0",
    }

    try:
        response = requests.get(url, params=params, headers=headers, timeout=timeout)
        response.raise_for_status()

        data = response.json()
        return data.get("display_name")
    except Exception as exc:
        print(f"[get_exact_location_from_lat_lng] error: {exc}")
        return None

# Querying Google Maps

import subprocess
import sys

def get_google_maps_stats_from_notebook(hotel_name: str) -> dict[str, str | float | int | None] | None:
    result = subprocess.run(
        [sys.executable, "get_google_rating.py", hotel_name],
        capture_output=True,
        text=True,
    )

    if result.returncode != 0:
        print(f"[google_rating] error: {result.stderr.strip()}")
        return None

    try:
        data = json.loads(result.stdout.strip())
        return data
    except Exception:
        print(f"[google_rating] invalid stdout: {result.stdout.strip()}")
        return None
    

def haversine_in_meters(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    # Convert degrees to radians
    lat1, lon1, lat2, lon2 = map(math.radians, [lat1, lon1, lat2, lon2])

    # Haversine formula
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = math.sin(dlat / 2) ** 2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2) ** 2
    c = 2 * math.asin(math.sqrt(a))
    r = 6371  # Radius of the Earth in km
    
    kilometers = c * r
    meters = kilometers * 1000
    return meters

# My RAPID API SkyScanner Functions

- Functions that are used for using the RAPID API SkyScanner website.
- Here's the api I'm using, it costs $8.99 to own a key.
- My keys are in my local laptop's env file, you should get your own.

## SETTINGS

In [ ]:
import datetime
import os
from typing_extensions import TypedDict  # not: from typing import TypedDict

import requests
from dotenv import load_dotenv

# Load environment variables from .env in the project root
load_dotenv()

rapidapi_key: str =  os.getenv("RAPIDAPI_KEY", "")
rapidapi_host: str = os.getenv("RAPIDAPI_HOST", "sky-scrapper.p.rapidapi.com")
headers = {
    "X-RapidAPI-Key": rapidapi_key,
    "X-RapidAPI-Host": rapidapi_host,
}

if not rapidapi_key:
    raise ValueError("RAPIDAPI_KEY is missing. Add it to your .env file.")

## Search Hotel Rates Function

In [ ]:
def SearchHotelRates(entityId: str, checkin_date: str, checkout_date: str):
    OutputDict = TypedDict(
        "OutputDict",
        {
            "partnerName": str, # ex: Booking.com
            "partnerLogo": str, # ex: "https://www.skyscanner.com.ph/images/websites/220x80/h_bc.png"
            "partnerId": str, # ex: "h_bc"
            "roomType": str, # ex: "Room"
            "roomPolicies": str, # ex: "Meals not included, Non-Refundable"
            "deeplink": str, # ex: "www.skyscanner.com.ph/...
            "rawPrice": float, # ex: 1486 (can have decimals sometimes)
            "rateBriefFeatures": list[str], # ex: ["Meals not included","Non-Refundable","Room"]
            "score": int, # ex: 1
            # -----
            "checkin_date": datetime.date,
            "checkout_date": datetime.date,
        },
        total=True,
        closed=True
    )

    url = f"https://sky-scrapper.p.rapidapi.com/api/v1/hotels/getHotelPrices?hotelId={entityId}&entityId={entityId}&checkin={checkin_date}&checkout={checkout_date}&adults=1&rooms=1&currency=PHP&market=en-US&countryCode=PH"
    try:
        response = requests.get(
            url,
            headers=headers,
        )
        if (response.status_code == 200):
            response_json = response.json()
            if (response_json.get("data")):
                data = response_json["data"]

                if (data.get("metaInfo")):
                    metaInfo = data["metaInfo"]
                    if (metaInfo.get("rates")):
                        rates = data["metaInfo"]["rates"]

                        OutputDictArray: list[OutputDict] = []
                        for rate in rates:
                            checkin_dt = datetime.date.fromisoformat(checkin_date)
                            checkout_dt = datetime.date.fromisoformat(checkout_date)
                            new_item: OutputDict = {
                                "partnerName": rate["partnerName"],
                                "partnerLogo": rate["partnerLogo"],
                                "partnerId": rate["partnerId"],
                                "roomType": rate["roomType"],
                                "roomPolicies": rate["roomPolicies"],
                                "deeplink": rate["deeplink"],
                                "rawPrice": rate["rawPrice"],
                                "rateBriefFeatures": rate["rateBriefFeatures"],
                                "score": rate["score"],
                                # ----
                                "checkin_date": checkin_dt,
                                "checkout_date": checkout_dt
                            }
                            OutputDictArray.append(new_item)
                        return OutputDictArray
                    else:
                        raise Exception("[SearchHotelRates] Raised Error: rates don't exist")
                else:
                    raise Exception("[SearchHotelRates] Raised Error: MetaInfo doesn't exist.")
        raise Exception("[SearchHotelRates] Raised Error: No data found in the response.")
    except Exception as e:
        print(f"[SearchHotelRates] major error: {e}")
        return None

## Search Hotel Details Function

In [ ]:
def SearchHotelDetails(entityId: str):
    OutputDict = TypedDict(
        "OutputDict",
        {
            "amenities": str,
            "description": str,
            "rating": float,
            "countNumber": int,
            # -----
            "checkin_time": datetime.time | None,
            "checkout_time": datetime.time | None,
        },
        total=True,
        closed=True
    )


    url = f"https://sky-scrapper.p.rapidapi.com/api/v1/hotels/getHotelDetails?hotelId={entityId}&entityId={entityId}&currency=PHP&market=en-US&countryCode=PH"
    try: 
        response = requests.get(
            url,
            headers=headers,
        )

        if (response.status_code == 200):
            response_json = response.json()
            if (response_json.get("data")):
                data = response_json["data"]
                
                checkin_time: datetime.time | None = None
                checkout_time: datetime.time | None = None
                description: str = ""
                if (data.get("goodToKnow")):
                    goodToKnow = data["goodToKnow"]
                    goodToKnowCheckinTime = goodToKnow["checkinTime"]
                    checkin_time_str = str(goodToKnowCheckinTime["time"]) # ex: "14:00"
                    checkin_time = datetime.time.fromisoformat(checkin_time_str)

                    goodToKnowCheckoutTime = goodToKnow["checkoutTime"]
                    checkout_time_str = str(goodToKnowCheckoutTime["time"]) # ex: "14:00"
                    checkout_time = datetime.time.fromisoformat(checkout_time_str)

                    description = goodToKnow["description"]["content"]
                
                amenities_content = ""
                if (data.get("amenities")):
                    amenities = data["amenities"]
                    if (amenities.get("content")):
                        content_arr = amenities["content"]
                        amenities_content = content_arr[0]["description"]

                rating = 0.0
                countNumber = 0
                if (data.get("reviewRatingSummary")):
                    reviewRatingSummary = data["reviewRatingSummary"]
                    rating_str:str = str(reviewRatingSummary["score"]) # ex: 7.8
                    rating = float(rating_str)

                    countNumber: int = reviewRatingSummary["countNumber"]

                final_check = checkin_time != None and checkout_time != None and amenities_content != "" and description != ""
                if (final_check):
                    new_hotel_data: OutputDict = {
                        "amenities": amenities_content,
                        "description": description,
                        "checkin_time": checkin_time,
                        "checkout_time": checkout_time,
                        "rating": rating,
                        "countNumber": countNumber,
                    }

                    return new_hotel_data
                
        raise Exception("[SearchHotelDetails] Raised Error: No data found in the response.")
    except Exception as e:
        print(f"[SearchHotelDetails] major error: {e}")
        return None

## Search Hotels Function Function + GetPOISOFHotel Function

In [ ]:
def SearchHotelsFunction(entityId: str, checkin_date: str, checkout_date: str):
    url = f"https://sky-scrapper.p.rapidapi.com/api/v1/hotels/searchHotels?entityId={entityId}&checkin={checkin_date}&checkout={checkout_date}&adults=1&rooms=1&limit=5&sorting=distance&currency=PHP&market=en-US&countryCode=PH"
    try: 
        response = requests.get(
            url,
            headers=headers,
        )

        if (response.status_code == 200):
            response_json = response.json()
            if (response_json.get("data")):
                data = response_json["data"]
                
                return data

        raise Exception("[SearchHotelsFunction] Raised Error: No data found in the response.")
    except Exception as e:
        print(f"[SearchHotelsFunction] major error: {e}")
        return None


def GetPOISOFHotel(entityId: str, checkin_date: str, checkout_date: str):
    OutputDict = TypedDict(
        "OutputDict",
        {
            "coordinate": list[float],
            "name": str,
            "id": str,
            "type": str,
            "sub_poi_type": str,
        },
        closed = True
    )

    OutputDictArray: list[OutputDict] = []

    data = SearchHotelsFunction(entityId, checkin_date, checkout_date)

    if (data != None):
        if (data.get("pois")):
            pois = data["pois"]

            for poi in pois:
                coordinate_raw: list[float] = poi.get("coordinate")
                latitude: float = 0.0
                longitude: float = 0.0

                if len(coordinate_raw) >= 2:
                    latitude = float(coordinate_raw[1])
                    longitude = float(coordinate_raw[0])

                new_output: OutputDict = {
                    "coordinate": [latitude, longitude],
                    "name": poi["name"],
                    "id": poi["id"],
                    "type": poi["type"],
                    "sub_poi_type": poi["sub_poi_type"],
                }

                OutputDictArray.append(new_output)
        return OutputDictArray    
    else:
        return None

## Search Hotels Function

In [ ]:

def SearchHotels(name_of_hotel: str):
    OutputDict = TypedDict(
        "OutputDict",
        {
            "hierarchy": str,
            "location": list[float],
            "score": int,
            "entityName": str,
            "entityId": str,
            "entityType": str,
            "suggestItem": str,
            "class": str,
            "pois": object,
        },
        total=True,
        closed=True
    )
    
    query = replace_space_with_percent20(name_of_hotel)
    url = f"https://sky-scrapper.p.rapidapi.com/api/v1/hotels/searchDestinationOrHotel?query={query}"
    try:
        response = requests.get(
            url,
            headers=headers,
        )

        if response.status_code == 200:
            response_json = response.json()
            if (response_json.get("data")):
                data = response_json["data"]
                if (len(data) > 0):
                    OutputDictArray: list[OutputDict] = []
                    for item in data:
                        location = item["location"]
                        location_parts = location.split(",")
                        x: float = float(location_parts[0].strip())
                        y: float = float(location_parts[1].strip())

                        if ("Philippines" in item["hierarchy"]) and ("Hotel" in item["class"]):
                            new_item: OutputDict = {
                                "hierarchy": item["hierarchy"],
                                "location": [x, y],
                                "score": item["score"],
                                "entityName": item["entityName"],
                                "entityId": item["entityId"],
                                "entityType": item["entityType"],
                                "suggestItem": item["suggestItem"],
                                "class": item["class"],
                                "pois": item["pois"]
                            } 
                            OutputDictArray.append(new_item)
                    
                    if len(OutputDictArray) == 0:
                        return None
                    else:
                        return OutputDictArray
                else:
                    print("No hotels found in the response.")
                    return None
        raise Exception("[SearchHotels] Raised Error: No data found in the response.")
    except Exception as e:
        print(f"[SearchHotels] major error: {e}")
        return None

----

# MAIN FUNCTION

- This is where I will be processing the rows.
- I'm using concurrent features because I don't have the patience to wait per row. It will run 50 threads, each processing one row, and there will always be an active 50 threads. 

In [ ]:
# SETTINGS:
debug_flag = True # if you want to test how it works Set it to True, otherwise False to run the whole process.

In [ ]:
import datetime
from random import randint

import concurrent.futures
import pandas as pd
from enum import Enum

from typing_extensions import TypedDict  # not: from typing import TypedDict

# Read a specific sheet from an .xlsm file
from pandas import DataFrame

inputDF: DataFrame = pd.read_excel("input-data/main.xlsm", sheet_name=0, engine="openpyxl") # type: ignore # ignore
inputDF = inputDF[inputDF["source"].str.contains("skyscanner", na=False)]  # contains the word skyscanner
inputDF = inputDF[inputDF["skyscrapper_hotel_id"].isna()]  # include only rows with no id. (Cause we will put them)
print(inputDF.columns)
print("How many are we processing?:", len(inputDF))


# TYPED DICTS
class Address(TypedDict, total=True):
    city: str
    second_level_nation_administrative_division: str
    first_level_nation_administrative_division: str
    country: str

# ENUMS
class PhilippinesRegion(str, Enum):
    NCR = "NCR"
    CAR = "CAR"
    REGION_I = "Region I"
    REGION_II = "Region II"
    REGION_III = "Region III"
    REGION_IV_A = "Region IV-A"
    REGION_IV_B = "Region IV-B"
    REGION_V = "Region V"
    REGION_VI = "Region VI"
    REGION_VII = "Region VII"
    REGION_VIII = "Region VIII"
    REGION_IX = "Region IX"
    REGION_X = "Region X"
    REGION_XI = "Region XI"
    REGION_XII = "Region XII"
    REGION_XIII = "Region XIII"
    BARMM = "BARMM"

class AreaType(str, Enum):
    URBAN = "Urban"
    SUBURBAN = "Suburban"
    RURAL = "Rural"

class Hotel(TypedDict, total=True, closed=True):
    skyscrapper_hotel_id: str
    name_of_hotel: str
    source: str
    booking_platform: str
    region: PhilippinesRegion
    latitude_and_longitude: str
    latitude: float
    longitude: float
    exact_location: str
    area_type: AreaType | None
    checkin_date: datetime.date
    checkin_time: datetime.time | None
    checkout_date: datetime.date
    checkout_time: datetime.time | None
    day_of_week: str
    room_type: str
    price: float
    current_season: None | str
    price_for_x_adults: int
    is_there_nearby_ShoppingMall: bool
    nearest_ShoppingMall_name: str
    ShoppingMall_latitude: float
    ShoppingMall_longtitude: float
    distance_from_nearest_ShoppingMall_in_meters: float
    is_near_an_airport: bool
    has_balcony: bool
    is_a_resort: bool
    rating_10_star_system: float
    rating: float
    num_of_reviews: int
    length_of_stay_days: int
    City: str
    SecondLevelNationAdministrativeDivision: str
    FirstLevelNationAdministrativeDivision: str
    Nation: str
    amenities: str
    stars: int
    description: str

output_results: list[Hotel] = []

# -----

def main_function(row: pd.Series, index: int):
    print(index,"-",row["name_of_hotel"])
    responseSearch = SearchHotels(row["name_of_hotel"])
    if (responseSearch != None):
        base_date = datetime.date.today()
        random_number = randint(1, 3)
        checkin_date_that_is_random = base_date + datetime.timedelta(days=(int(index) % 30) + 1)
        checkout_date_that_is_random = checkin_date_that_is_random + datetime.timedelta(days=random_number)
        checkin_date_that_is_random = checkin_date_that_is_random.isoformat()
        checkout_date_that_is_random = checkout_date_that_is_random.isoformat()

        for output in responseSearch:
            if str(output["entityId"]) != "":
                print("✔️ EntityID:", output["entityId"])
                print(f"🔎 Querying POIs for hotel: {output['entityName']} with ID: {output['entityId']} | {checkin_date_that_is_random} - {checkout_date_that_is_random} ")
                responsePOISOfHotel = GetPOISOFHotel(entityId=output["entityId"], checkin_date=checkin_date_that_is_random, checkout_date=checkout_date_that_is_random)
                if (responsePOISOfHotel != None):
                    print("✔️ GetPOISOFHotel response exists")
                    print(responsePOISOfHotel)
                else:
                    print("❌ GetPOISOFHotel response does not exist")

                print(f"🔎 Querying Google Maps for hotel: {output['entityName']} to get rating and number of reviews...")
                google_stats = get_google_maps_stats_from_notebook(output["entityName"])

                responseHotelDetails = SearchHotelDetails(entityId=output["entityId"])
                if (responseHotelDetails != None):
                    print("✔️ responseHotelDetails response exists")
                    print(responseHotelDetails)
                else:
                    print("❌ responseHotelDetails response does not exist")
            
                print(f"🔎 Querying rates for hotel: {output['entityName']} with ID: {output['entityId']} | {checkin_date_that_is_random} - {checkout_date_that_is_random} ")
                responseHotelRates = SearchHotelRates(entityId=output["entityId"], checkin_date=checkin_date_that_is_random, checkout_date=checkout_date_that_is_random)
                if (responseHotelRates != None):
                    print("✔️ responseHotelRates response exists")
                    print(responseHotelRates)
                else:
                    print("❌ responseHotelRates response does not exist")

                if (responseHotelRates != None):
                    print("✔️ hotel rates exist:")
                    location = output["hierarchy"].split("|")
                    city = location[0].strip()
                    second_level_nation_administrative_division = location[1].strip()
                    
                    latitude: float = output["location"][0]
                    longtitude: float = output["location"][1]

                    exact_location: str = str(get_exact_location_from_lat_lng(latitude, longtitude))
                    region = get_ph_region(exact_location)
                    region_enum: PhilippinesRegion = PhilippinesRegion(region)
                    first_level_nation_administrative_division = exact_location.split(",")[-3].strip() # exact location gives me this: 'exact_location': 'EV English Academy, President Roxas Street, Villa Aurora, Kasambagan, Cebu City, Central Visayas, 6000, Philippines' Central Visayas is the firstLevelNationAdministrationDivision, get that.
                    
                    for rate in responseHotelRates:
                        amenities: str = rate["roomPolicies"]
                        print(amenities)

                        rating = 0
                        num_of_reviews = 0
                        if (google_stats != None):
                            print("✔️ Google Maps stats found:")
                            latitude_value = google_stats.get("latitude")
                            longtitude_value = google_stats.get("longtitude")
                            rating_value = google_stats.get("rating")
                            review_count_value = google_stats.get("review_count")
                            if isinstance(latitude_value, (int, float)):
                                latitude = float(latitude_value)
                            if isinstance(longtitude_value, (int, float)):
                                longtitude = float(longtitude_value)
                            if isinstance(rating_value, (int, float)):
                                rating = float(rating_value)
                            if isinstance(review_count_value, (int, float)):
                                num_of_reviews = int(review_count_value)
                        if (rating == 0) or (num_of_reviews == 0):
                            if (responseHotelDetails != None):
                                rating = responseHotelDetails["rating"]
                                num_of_reviews = responseHotelDetails["countNumber"]

                        firstShoppingMallFoundName: str = "N/A"
                        firstShoppingMallLatitude: float = 0.0
                        firstShoppingMallLongtitude: float = 0.0
                        distance_from_nearest_ShoppingMall_in_meters: float = 0.0
                        is_near_an_airport = False
                        if (responsePOISOfHotel != None):
                            for poi in responsePOISOfHotel:
                                if (poi["sub_poi_type"] == "Airport"):
                                    is_near_an_airport = True
                                if (poi["sub_poi_type"] == "ShoppingMall"):
                                    firstShoppingMallFoundName = poi["name"]
                                    firstShoppingMallLatitude = poi["coordinate"][0]
                                    firstShoppingMallLongtitude = poi["coordinate"][1]
                                    distance_from_nearest_ShoppingMall_in_meters = haversine_in_meters(latitude, longtitude, firstShoppingMallLatitude, firstShoppingMallLongtitude)    
                                    break    
                            if is_near_an_airport == False:
                                for poi in responsePOISOfHotel:
                                    if (poi["sub_poi_type"] == "Airport"):
                                        is_near_an_airport = True
                                        break

                        raw_checkin_date = rate["checkin_date"]
                        if isinstance(raw_checkin_date, datetime.datetime):
                            checkin_date_value = raw_checkin_date.date()
                        else:
                            checkin_date_value = datetime.date.fromisoformat(str(raw_checkin_date))
                        raw_checkout_date = rate["checkout_date"]
                        if isinstance(raw_checkout_date, datetime.datetime):
                            checkout_date_value = raw_checkout_date.date()
                        else:
                            checkout_date_value = datetime.date.fromisoformat(str(raw_checkout_date))
                        checkin_time: datetime.time | None = None
                        checkout_time: datetime.time | None = None
                        description: str = ""
                        if (responseHotelDetails != None):
                            checkin_time = responseHotelDetails["checkin_time"]
                            checkout_time = responseHotelDetails["checkout_time"]
                            description = responseHotelDetails["description"]
                            amenities = amenities + ", " + responseHotelDetails["amenities"]
                            # replace the spaces that aren't after the , with _ and lower case everything
                            amenities_parts = amenities.split(", ")
                            amenities_parts = [part.replace(" ", "_") for part in amenities_parts]
                            amenities = ", ".join(amenities_parts).lower()
                            # also remove duplicates in amenities
                            amenities_list = amenities.split(", ")
                            amenities_list = list(set(amenities_list))
                            amenities = ", ".join(amenities_list)

                        is_there_nearby_shopping_mall = (firstShoppingMallFoundName != "N/A")
                        length_of_stay_days = (checkout_date_value - checkin_date_value).days

                        resort_keywords = ["ocean", "beach", "resort", "seaside", "seaview", "island"]
                        has_resort_keyword = any(keyword in amenities for keyword in resort_keywords) or any(keyword in description for keyword in resort_keywords)

                        balcony_keywords = ["balcony", "terrace", "veranda", "patio", "deck"]
                        has_balcony = any(keyword in amenities for keyword in balcony_keywords) or any(keyword in description for keyword in balcony_keywords)

                        HotelData: Hotel = {
                            "skyscrapper_hotel_id": str(output["entityId"]),
                            "name_of_hotel": str(output["entityName"]),
                            "source": str(rate["deeplink"]),
                            "booking_platform": str(rate["partnerName"]),
                            "latitude_and_longitude": f"{latitude}, {longtitude}",
                            "latitude": float(latitude),
                            "longitude": float(longtitude),
                            "exact_location": str(exact_location),
                            "region": region_enum,
                            "area_type": None,
                            "checkin_date": checkin_date_value,
                            "checkin_time": checkin_time,
                            "checkout_date": checkout_date_value,
                            "checkout_time": checkout_time,
                            "day_of_week": str(checkin_date_value.strftime("%A")).capitalize(),
                            "room_type": str(rate["roomType"]),
                            "price": float(rate["rawPrice"]),
                            "current_season": None,
                            "price_for_x_adults": 1,
                            "has_balcony": has_balcony,
                            "is_a_resort": has_resort_keyword,
                            "is_there_nearby_ShoppingMall": is_there_nearby_shopping_mall,
                            "nearest_ShoppingMall_name": firstShoppingMallFoundName,
                            "ShoppingMall_latitude": float(firstShoppingMallLatitude),
                            "ShoppingMall_longtitude": float(firstShoppingMallLongtitude),
                            "distance_from_nearest_ShoppingMall_in_meters": float(distance_from_nearest_ShoppingMall_in_meters),
                            "is_near_an_airport": is_near_an_airport,
                            "rating_10_star_system": float(rating * 2),
                            "rating": float(rating),
                            "num_of_reviews": int(num_of_reviews),
                            "length_of_stay_days": int(length_of_stay_days),
                            "City": str(city),
                            "SecondLevelNationAdministrativeDivision": str(second_level_nation_administrative_division),
                            "FirstLevelNationAdministrativeDivision": str(first_level_nation_administrative_division),
                            "Nation": "Philippines",
                            "amenities": str(amenities),
                            "stars": int(rating),
                            "description": description
                        }
                        print("✅ final output:", HotelData)
                        return HotelData
                else:
                    print("❌ [RUN FAILED] no rates found")
                    return None
                
    print("❌ [MAJOR FAILURE]")
    return None

if (debug_flag):
    shuffled_df = inputDF.sample(frac=1)
    with concurrent.futures.ThreadPoolExecutor(max_workers=50) as executor:
        futures: list[concurrent.futures.Future[Hotel | None]] = []
        for index, (_, row) in enumerate(shuffled_df.head(5).iterrows()):
            futures.append(executor.submit(main_function, row, index))

        for future in concurrent.futures.as_completed(futures):
            result = future.result()
            if result is not None:
                output_results.append(result)
else:
    with concurrent.futures.ThreadPoolExecutor(max_workers=50) as executor:
        futures: list[concurrent.futures.Future[Hotel | None]] = []
        for index, (_, row) in enumerate(inputDF.iterrows()):
            futures.append(executor.submit(main_function, row, index))

        for future in concurrent.futures.as_completed(futures):
            result = future.result()
            if result is not None:
                output_results.append(result)        

# Save the output results to a csv file
print("\n", "PROCESS FINISHED, turning output_results into DataFrame, then into CSV (output-data/hotel_data.csv)", "\n")
outputDF = pd.DataFrame(output_results)
outputDF.to_csv("output-data/hotel_data.csv", index=False)

Index(['skyscrapper_hotel_id', 'name_of_hotel', 'source', 'booking_platform',
       'latitude_and_longtitude', 'latitude', 'longtitude', 'exact_location',
       'region', 'area_type', 'checkin_date', 'checkin_time', 'checkout_date',
       'checkout_time', 'day_of_the_week', 'room_type', 'price',
       'current_season', 'price_for_x_adults', 'has_balcony', 'is_a_resort',
       'is_there_nearby_ShoppingMall', 'nearest_ShoppingMall_name',
       'ShoppingMall_latitude', 'ShoppingMall_longtitude',
       'distance_from_nearest_ShoppingMall_in_meters', 'is_near_an_airport',
       'rating_10_star_system', 'rating', 'num_of_reviews',
       'length_of_stay_days', 'City',
       'SecondLevelNationAdministrativeDivision',
       'FirstLevelNationAdministrativeDivision', 'Nation', 'amenities',
       'stars', 'description'],
      dtype='str')
How many are we processing?: 3816
0 - CF Haven - Near IT Park, Ayala, SM
1 - East Bay Residences
2 - The Mood Shift
3 - Chelsea Tower 1
4 - Anuva Re